In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import sys, subprocess, glob, os
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rouge-score"], check=True)

# experiment_checkpoint folder dhundho (dataset jahan bhi unzip hua ho)
hits = glob.glob("/kaggle/input/**/experiment_checkpoint/experiment_metadata.json", recursive=True)
print(hits)
assert hits, "experiment_checkpoint nahi mila - fragment-phase1 dataset Add Input se joda hai?"
CK = os.path.dirname(hits[0])
print("CHECKPOINT:", CK)
for root, _, files in os.walk(CK):
    print(root.replace(CK, "."), files)

# adapters bhi dikh rahe hain ya nahi
print(glob.glob("/kaggle/input/**/*.safetensors", recursive=True))

['/kaggle/input/datasets/khushisoni007/fragment-phase1/experiment_checkpoint/experiment_metadata.json']
CHECKPOINT: /kaggle/input/datasets/khushisoni007/fragment-phase1/experiment_checkpoint
. ['adapters_manifest.json', 'experiment_metadata.json', 'tone_independent_eval.jsonl']
./base_results ['json_base.jsonl', 'samsum_base.jsonl', 'sql_base.jsonl', 'tone_base.jsonl']
./evaluation_datasets []
./evaluation_datasets/json ['data-00000-of-00001.arrow', 'dataset_info.json', 'state.json']
./evaluation_datasets/samsum ['data-00000-of-00001.arrow', 'dataset_info.json', 'state.json']
./evaluation_datasets/sql ['data-00000-of-00001.arrow', 'dataset_info.json', 'state.json']
./evaluation_datasets/tone ['data-00000-of-00001.arrow', 'dataset_info.json', 'state.json']
./solo_results ['json_solo.jsonl', 'samsum_solo.jsonl', 'solo_summary.json', 'sql_solo.jsonl', 'tone_solo.jsonl']
['/kaggle/input/datasets/khushisoni007/ai-fragment/adapter_model.safetensors', '/kaggle/input/datasets/khushisoni007/ai-

In [2]:
import json, re, statistics as st
from collections import Counter
from rouge_score import rouge_scorer

R = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
TASKS = ["json", "tone", "sql", "samsum"]

def load(mode, task):
    p = f"{CK}/{mode}_results/{task}_{mode}.jsonl"
    return {r["example_id"]: r for r in (json.loads(l) for l in open(p) if l.strip())}

# --- metrics ---
def rougeL(ref, pred): return R.score(ref, pred)["rougeL"].fmeasure

def norm_sql(s):
    s = re.sub(r"```(sql)?", "", s).strip()
    s = s.split("\n\n")[0]
    return re.sub(r"\s+", " ", s).strip().rstrip(";").lower()

def first_json(t):
    t = re.sub(r"```(json)?", "", t); i = t.find("{")
    if i < 0: return None
    try: return json.JSONDecoder().raw_decode(t[i:])[0]
    except Exception: return None

def flat(o, p=""):
    out = []
    if isinstance(o, dict):
        for k, v in o.items(): out += flat(v, f"{p}.{k}")
    elif isinstance(o, list):
        for v in o: out += flat(v, p + "[]")
    elif o not in (None, "", []): out.append((p, str(o).strip().lower()))
    return out

def json_f1(ref, pred):
    ref = json.loads(ref) if isinstance(ref, str) else ref
    pr = first_json(pred)
    a = Counter(flat(pr)) if pr is not None else Counter(); b = Counter(flat(ref))
    tp = sum((a & b).values())
    if tp == 0: return 0.0
    p, q = tp / sum(a.values()), tp / sum(b.values())
    return 2 * p * q / (p + q)

EMPATHY = re.compile(r"\b(sorry|apolog\w*|understand\w*|frustrat\w*|inconvenien\w*)\b", re.I)

def score(task, r):
    ref, pred = r["reference"], r["prediction"]
    if task == "json":   return {"f1": json_f1(ref, pred), "parsed": float(first_json(pred) is not None)}
    if task == "sql":    return {"em": float(norm_sql(pred) == norm_sql(ref)), "rougeL": rougeL(ref, pred)}
    if task == "samsum": return {"rougeL": rougeL(ref, pred), "len": len(pred)}
    if task == "tone":   return {"rougeL": rougeL(ref, pred), "empathy": float(bool(EMPATHY.search(pred))), "len": len(pred)}

# --- base vs solo, sirf woh rows jo dono mein successful hain ---
SCORES, KEEP_IDS = {}, {}
for task in TASKS:
    base, solo = load("base", task), load("solo", task)
    ok = [i for i in base if i in solo
          and "error" not in base[i] and "error" not in solo[i]
          and base[i]["prediction"].strip() and solo[i]["prediction"].strip()]
    KEEP_IDS[task] = ok
    for mode, d in [("base", base), ("solo", solo)]:
        rows = [score(task, d[i]) for i in ok]
        SCORES[(task, mode)] = {k: round(st.mean(x[k] for x in rows), 4) for k in rows[0]}
    print(f"{task.upper():7} n={len(ok):3} | BASE {SCORES[(task,'base')]} | SOLO {SCORES[(task,'solo')]}")

# --- save ---
os.makedirs("/kaggle/working/scores", exist_ok=True)
json.dump({"scores": {f"{t}|{m}": v for (t, m), v in SCORES.items()}, "keep_ids": KEEP_IDS},
          open("/kaggle/working/scores/phase1_scores.json", "w"), indent=2)
print("\nsaved -> /kaggle/working/scores/phase1_scores.json")

JSON    n= 70 | BASE {'f1': 0.369, 'parsed': 0.5857} | SOLO {'f1': 0.6615, 'parsed': 0.8286}
TONE    n=150 | BASE {'rougeL': 0.2192, 'empathy': 0.5667, 'len': 566.62} | SOLO {'rougeL': 0.189, 'empathy': 0.6333, 'len': 337.2867}
SQL     n=204 | BASE {'em': 0.0098, 'rougeL': 0.4996} | SOLO {'em': 0.1127, 'rougeL': 0.7373}
SAMSUM  n=150 | BASE {'rougeL': 0.2536, 'len': 357.7467} | SOLO {'rougeL': 0.3017, 'len': 288.8333}

saved -> /kaggle/working/scores/phase1_scores.json


In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # torch import se PEHLE
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=True)

import torch, transformers, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
print("GPU:", torch.cuda.get_device_name(0))

torch 2.11.0+cu130 | transformers 5.20.2 | peft 0.21.2
GPU: Tesla T4


In [5]:
import json, hashlib, shutil, glob
from pathlib import Path
from huggingface_hub import snapshot_download
from peft import LoraConfig

MAN = json.load(open(f"{CK}/adapters_manifest.json"))
BASE = "mistralai/Mistral-7B-Instruct-v0.2"
AD = Path("/kaggle/working/adapters"); AD.mkdir(parents=True, exist_ok=True)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""): h.update(c)
    return h.hexdigest()

# in-house: ai-fragment ki safetensors ko hash se pehchano
local = {sha256(p): p for p in glob.glob("/kaggle/input/**/ai-fragment/*.safetensors", recursive=True)}
for name in ["json", "tone"]:
    src = local.get(MAN[name]["sha256"])
    assert src, f"{name}: matching file nahi mili (hash {MAN[name]['sha256'][:12]})"
    (AD / name).mkdir(exist_ok=True)
    shutil.copy2(src, AD / name / "adapter_model.safetensors")
    LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
               target_modules=MAN[name]["target_modules"], base_model_name_or_path=BASE).save_pretrained(AD / name)
    print(name, "<-", src)

# external: pinned commit
for name in ["sql", "samsum"]:
    snapshot_download(MAN[name]["repo"], revision=MAN[name]["revision"], local_dir=str(AD / name),
                      allow_patterns=["adapter_config.json", "adapter_model.safetensors"])

for name in ["json", "tone", "sql", "samsum"]:
    ok = sha256(AD / name / "adapter_model.safetensors") == MAN[name]["sha256"]
    print(f"{name:7} sha256 {'MATCH' if ok else 'MISMATCH'}")
    assert ok
print("ALL 4 ADAPTERS READY")

json <- /kaggle/input/datasets/khushisoni007/ai-fragment/adapter_model.safetensors
tone <- /kaggle/input/datasets/khushisoni007/ai-fragment/adapter_model_tone.safetensors


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

json    sha256 MATCH
tone    sha256 MATCH
sql     sha256 MATCH
samsum  sha256 MATCH
ALL 4 ADAPTERS READY


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

REV = json.load(open(f"{CK}/experiment_metadata.json"))["base_revision"]
tok = AutoTokenizer.from_pretrained(BASE, revision=REV)
if tok.pad_token is None: tok.pad_token = tok.eos_token

q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                       bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
base_model = AutoModelForCausalLM.from_pretrained(BASE, revision=REV, quantization_config=q,
                                                  device_map="auto", dtype=torch.float16)
model = PeftModel.from_pretrained(base_model, str(AD / "json"), adapter_name="json", is_trainable=False)
for n in ["tone", "sql", "samsum"]:
    model.load_adapter(str(AD / n), adapter_name=n, is_trainable=False)
model.eval()
print("adapters:", list(model.peft_config), "| GPU GB:", round(torch.cuda.memory_allocated() / 1e9, 2))

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

adapters: ['json', 'tone', 'sql', 'samsum'] | GPU GB: 4.54


In [7]:
from datasets import load_from_disk
import statistics as st

tone_eval = load_from_disk(f"{CK}/evaluation_datasets/tone")

def tone_ids(ex):
    user = f"Subject: {(ex['subject'] or '(no subject)').strip()}\n\n{ex['body']}"
    prompt = tok.apply_chat_template([{"role": "user", "content": user}], tokenize=False, add_generation_prompt=True)
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    a = tok(ex["answer"].strip(), add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
    return p, a

@torch.no_grad()
def answer_loss(p, a):
    ids = torch.tensor([p + a], device=model.device)
    lab = ids.clone(); lab[0, :len(p)] = -100
    return model(input_ids=ids, labels=lab).loss.item()

def run_tone_loss(mode):
    out = []
    for ex in tone_eval:
        p, a = tone_ids(ex)
        try:
            if mode == "base":
                with model.disable_adapter(): out.append(answer_loss(p, a))
            else:
                model.set_adapter(mode); out.append(answer_loss(p, a))
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); out.append(None)
    return out

TONE_LOSS = {m: run_tone_loss(m) for m in ["base", "tone"]}
pairs = [(b, t) for b, t in zip(TONE_LOSS["base"], TONE_LOSS["tone"]) if b is not None and t is not None]
print(f"n={len(pairs)} | BASE loss {st.mean(b for b,_ in pairs):.4f} | TONE loss {st.mean(t for _,t in pairs):.4f} "
      f"| rows where tone < base: {sum(t < b for b, t in pairs)}/{len(pairs)}")
json.dump(TONE_LOSS, open("/kaggle/working/scores/tone_loss.json", "w"))

n=150 | BASE loss 2.8320 | TONE loss 1.6628 | rows where tone < base: 149/150


In [8]:
import re
from datasets import load_dataset

ni = load_dataset("Lots-of-LoRAs/task1572_samsum_summary")
print(ni)
print("\n--- ADAPTER KA TRAINING INPUT FORMAT ---\n" + repr(ni["train"][0]["input"][:1500]))
print("\nOUTPUT:", ni["train"][0]["output"])

# kya hamare 150 test dialogues adapter ke training data mein the? (leakage check)
sams = load_from_disk(f"{CK}/evaluation_datasets/samsum")
norm = lambda s: re.sub(r"\s+", " ", s).strip()
ni_text = " ||| ".join(norm(x) for split in ni for x in ni[split]["input"])
hits = sum(norm(d)[:150] in ni_text for d in sams["dialogue"])
print(f"\nOVERLAP: {hits}/{len(sams)} test dialogues adapter ke data mein mile")

README.md:   0%|          | 0.00/2.77k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  114kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 27.7kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/valid-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 27.2kB            

data/valid-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating valid split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'id'],
        num_rows: 160
    })
    test: Dataset({
        features: ['input', 'output', 'id'],
        num_rows: 20
    })
    valid: Dataset({
        features: ['input', 'output', 'id'],
        num_rows: 20
    })
})

--- ADAPTER KA TRAINING INPUT FORMAT ---
"Definition: In this task, you are given a conversation, and your task is to generate a summary from the information present in the given conversation. Generate a summary in such a way that the context should be present in the conversation. It should cover the complete context of the conversation.\n\nPositive Example 1 -\nInput: Lucas: Hey! How was your day?, Demi: Hey there! , Demi: It was pretty fine, actually, thank you!, Demi: I just got promoted! :D, Lucas: Whoa! Great news!, Lucas: Congratulations!, Lucas: Such a success has to be celebrated., Demi: I agree! :D, Demi: Tonight at Death & Co.?, Lucas: Sure!, Lucas: See you there at 10pm?, Demi: Ye

In [9]:
s = ni["train"][0]["input"]
i = s.rfind("Input: ")              # last example = jo model ko complete karna hai
j = s.find("\nOutput", i)
NI_PREFIX, NI_SUFFIX = s[:i + len("Input: ")], s[j:]
print("PREFIX ka end:", repr(NI_PREFIX[-200:]))
print("SUFFIX:", repr(NI_SUFFIX))

def ni_dialogue(d):                 # "A: hi\r\nB: yo" -> "A: hi, B: yo"
    return ", ".join(l.strip() for l in d.splitlines() if l.strip())

def samsum_prompt(ex):
    user = NI_PREFIX + ni_dialogue(ex["dialogue"]) + NI_SUFFIX
    return tok.apply_chat_template([{"role": "user", "content": user}], tokenize=False, add_generation_prompt=True)

print("\n--- SAMPLE PROMPT (end) ---\n", samsum_prompt(sams[0])[-400:])

hits = sum(norm(ni_dialogue(d))[:150] in ni_text for d in sams["dialogue"])
print(f"\nOVERLAP (sahi format mein): {hits}/{len(sams)}")

PREFIX ka end: ' oh, cool you can make it when we go, Jade: yea, yesss, Wayne: :D\nOutput: The trip Jade was going to go on was postponed. Now Wayne will be able to go to.\n\nNow complete the following example -\nInput: '
SUFFIX: '\nOutput: \n\n'

--- SAMPLE PROMPT (end) ---
 ol you can make it when we go, Jade: yea, yesss, Wayne: :D
Output: The trip Jade was going to go on was postponed. Now Wayne will be able to go to.

Now complete the following example -
Input: Claire: <file_photo>, Kim: Looks delicious..., Linda: No way... Look what I'm cooking right now:, Linda: <file_photo>, Claire: hahahaha, Kim: Curry dream team, Claire: Enjoy your dinner :*
Output: 

 [/INST]

OVERLAP (sahi format mein): 0/150


In [10]:
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
R = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
os.makedirs("/kaggle/working/results", exist_ok=True)

@torch.no_grad()
def gen(prompt, max_new=100):
    x = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    y = model.generate(**x, max_new_tokens=max_new, do_sample=False,
                       pad_token_id=tok.eos_token_id, eos_token_id=tok.eos_token_id)
    return tok.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()

SAMSUM_NI = {}
for mode in ["base", "samsum"]:
    path = f"/kaggle/working/results/samsum_ni_{mode}.jsonl"
    done = {}
    if os.path.exists(path):
        done = {r["example_id"]: r for r in map(json.loads, open(path))}
    for k, ex in enumerate(tqdm(sams, desc=mode)):
        eid = str(ex.get("id", k))
        if eid in done: continue
        p = samsum_prompt(ex)
        if mode == "base":
            with model.disable_adapter(): pred = gen(p)
        else:
            model.set_adapter("samsum"); pred = gen(p)
        done[eid] = {"example_id": eid, "prediction": pred, "reference": ex["summary"]}
        with open(path, "a") as f: f.write(json.dumps(done[eid]) + "\n")      # resume-safe
    SAMSUM_NI[mode] = list(done.values())
    sc = [R.score(r["reference"], r["prediction"])["rougeL"].fmeasure for r in SAMSUM_NI[mode]]
    print(f"{mode.upper():7} ROUGE-L {st.mean(sc):.4f} | avg len {st.mean(len(r['prediction']) for r in SAMSUM_NI[mode]):.0f}")

print("\nexample:\n BASE :", SAMSUM_NI["base"][0]["prediction"][:200],
      "\n SOLO :", SAMSUM_NI["samsum"][0]["prediction"][:200],
      "\n REF  :", SAMSUM_NI["samsum"][0]["reference"][:200])

base:   0%|          | 0/150 [00:00<?, ?it/s]

BASE    ROUGE-L 0.3115 | avg len 273


samsum:   0%|          | 0/150 [00:00<?, ?it/s]

SAMSUM  ROUGE-L 0.4179 | avg len 152

example:
 BASE : Linda is cooking curry, while Claire looks at a photo of a delicious-looking dish. They both laugh and Claire wishes Linda a enjoyable dinner. 
 SOLO : Linda is cooking curry. Claire is eating a delicious looking dish. 
 REF  : Both Claire and Linda are making curry for dinner. 


In [11]:
import torch, os
from safetensors.torch import load_file, save_file
from peft import LoraConfig

NAMES = ["json", "tone", "sql", "samsum"]
W = {n: load_file(str(AD / n / "adapter_model.safetensors")) for n in NAMES}
SCALE = 32 / 16          # har adapter ka alpha/r
R_MERGED = 64            # 4 adapters x r16 (q,k,v par sab 4 hain)
SUF_A, SUF_B = ".lora_A.weight", ".lora_B.weight"

def build_concat(weights, out_dir):
    """weights = {adapter: w}. Merged update = sum_i w_i * 2.0 * B_i A_i (exact)."""
    prefixes = sorted({k[:-len(SUF_A)] for n in weights for k in W[n] if k.endswith(SUF_A)})
    assert prefixes, "lora_A keys nahi mile - key format check karo"
    out = {}
    for pre in prefixes:
        As, Bs = [], []
        for n, w in weights.items():
            if pre + SUF_A not in W[n]:
                continue                                   # missing module = zero update
            As.append(W[n][pre + SUF_A].float())
            Bs.append(W[n][pre + SUF_B].float() * (w * SCALE))
        A, B = torch.cat(As, 0), torch.cat(Bs, 1)
        pad = R_MERGED - A.shape[0]
        assert pad >= 0
        if pad:
            A = torch.cat([A, A.new_zeros(pad, A.shape[1])], 0)
            B = torch.cat([B, B.new_zeros(B.shape[0], pad)], 1)
        out[pre + SUF_A], out[pre + SUF_B] = A.contiguous(), B.contiguous()
    os.makedirs(out_dir, exist_ok=True)
    save_file(out, f"{out_dir}/adapter_model.safetensors")
    mods = sorted({p.split(".")[-1] for p in prefixes})
    LoraConfig(r=R_MERGED, lora_alpha=R_MERGED, lora_dropout=0.0, bias="none",        # alpha=r -> scale 1
               task_type="CAUSAL_LM", target_modules=mods, base_model_name_or_path=BASE).save_pretrained(out_dir)
    return out_dir

def chat(user):
    return tok.apply_chat_template([{"role": "user", "content": user}], tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def last_logits(adapter, prompt):
    model.set_adapter(adapter)
    x = tok(prompt, return_tensors="pt").to(model.device)
    return model(**x).logits[0, -1].float()

# ---- SANITY: single-adapter "merge" == original adapter ----
probe = chat(load_from_disk(f"{CK}/evaluation_datasets/sql")[0]["input"])
for n in NAMES:
    d = build_concat({n: 1.0}, f"/kaggle/working/merged/_single_{n}")
    model.load_adapter(d, adapter_name="chk", is_trainable=False)
    a, b = last_logits(n, probe), last_logits("chk", probe)
    print(f"{n:7} max|Δlogit| = {(a - b).abs().max().item():.4f} | same top token: {a.argmax().item() == b.argmax().item()}")
    model.delete_adapter("chk")

json    max|Δlogit| = 0.0234 | same top token: True


/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:2119: UserWarning: Adapter chk was active which is now deleted. Setting active adapter to json.
  warnings.warn(


tone    max|Δlogit| = 0.0127 | same top token: True
sql     max|Δlogit| = 0.0195 | same top token: True
samsum  max|Δlogit| = 0.0176 | same top token: True


In [13]:
import json, re, statistics as st
from collections import Counter
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
R = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

EVAL = {t: load_from_disk(f"{CK}/evaluation_datasets/{t}") for t in ["json", "sql", "samsum", "tone"]}
SUB = {"json": [i for i in range(31) if i != 56][:30], "sql": list(range(60)),
       "samsum": list(range(50)), "tone": list(range(len(EVAL["tone"])))}

# ---------- prompts (Phase 1 jaisa hi) ----------
JSON_TMPL = """Extract the information from the text as JSON that follows the schema.
Output JSON only, no explanation.

Instruction: Extract all the information that the schema asks for.

Schema:
{schema}

Text:
{text}"""
PROMPT = {   # task -> (prompt, add_special_tokens, max_new_tokens)   [Phase 1 settings]
    "json":   lambda ex: (chat(JSON_TMPL.format(schema=ex["schema"], text=ex["text"])), True, 400),
    "sql":    lambda ex: (chat(ex["input"]), True, 200),
    "samsum": lambda ex: (samsum_prompt(ex), False, 100),
}
REF = {"json": lambda ex: ex["item"],
       "sql": lambda ex: ex["output"][0] if isinstance(ex["output"], list) else ex["output"],
       "samsum": lambda ex: ex["summary"]}

@torch.no_grad()
def generate(prompt, add_special, max_new):
    x = tok(prompt, return_tensors="pt", add_special_tokens=add_special).to(model.device)
    y = model.generate(**x, max_new_tokens=max_new, do_sample=False,
                       pad_token_id=tok.eos_token_id, eos_token_id=tok.eos_token_id)
    return tok.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# ---------- scoring ----------
def first_json(t):
    t = re.sub(r"```(json)?", "", t); i = t.find("{")
    if i < 0: return None
    try: return json.JSONDecoder().raw_decode(t[i:])[0]
    except Exception: return None
def flat(o, p=""):
    if isinstance(o, dict): return [x for k, v in o.items() for x in flat(v, f"{p}.{k}")]
    if isinstance(o, list): return [x for v in o for x in flat(v, p + "[]")]
    return [] if o in (None, "", []) else [(p, str(o).strip().lower())]
def json_f1(ref, pred):
    ref = json.loads(ref) if isinstance(ref, str) else ref
    pr = first_json(pred); a = Counter(flat(pr)) if pr is not None else Counter(); b = Counter(flat(ref))
    tp = sum((a & b).values())
    return 0.0 if tp == 0 else 2 * (tp / sum(a.values())) * (tp / sum(b.values())) / (tp / sum(a.values()) + tp / sum(b.values()))
def norm_sql(s):
    s = re.sub(r"```(sql)?", "", s).strip().split("\n\n")[0]
    return re.sub(r"\s+", " ", s).strip().rstrip(";").lower()
def score_pred(task, ref, pred):
    if task == "json": return {"main": json_f1(ref, pred)}
    rl = R.score(ref, pred)["rougeL"].fmeasure
    if task == "sql": return {"main": rl, "em": float(norm_sql(pred) == norm_sql(ref))}
    return {"main": rl}

# ---------- Base / Solo reference scores (Phase 1 files, position-wise) ----------
def lines(p): return [json.loads(l) for l in open(p) if l.strip()]
FILES = {("json", "base"): f"{CK}/base_results/json_base.jsonl", ("json", "solo"): f"{CK}/solo_results/json_solo.jsonl",
         ("sql", "base"):  f"{CK}/base_results/sql_base.jsonl",  ("sql", "solo"):  f"{CK}/solo_results/sql_solo.jsonl",
         ("samsum", "base"): "/kaggle/working/results/samsum_ni_base.jsonl",
         ("samsum", "solo"): "/kaggle/working/results/samsum_ni_samsum.jsonl"}
REFSC = {}
for (task, mode), p in FILES.items():
    rows = lines(p); assert len(rows) == len(EVAL[task]), (task, mode, len(rows))
    assert rows[SUB[task][0]]["reference"] == REF[task](EVAL[task][SUB[task][0]]), f"order mismatch {task}"
    REFSC[(task, mode)] = [score_pred(task, r["reference"], r["prediction"]) for r in rows]
TONE_LOSS = json.load(open("/kaggle/working/scores/tone_loss.json"))

# ---------- evaluate one (merged) adapter ----------
def eval_adapter(name):
    model.set_adapter(name); res = {}
    for task in ["json", "sql", "samsum"]:
        out = []
        for i in tqdm(SUB[task], desc=f"{name}:{task}"):
            ex = EVAL[task][i]; p, sp, mx = PROMPT[task](ex)
            try: pred = generate(p, sp, mx)
            except torch.cuda.OutOfMemoryError: torch.cuda.empty_cache(); pred = ""
            out.append({"i": i, "pred": pred, **score_pred(task, REF[task](ex), pred)})
        res[task] = out
    res["tone"] = [{"i": i, "loss": answer_loss(*tone_ids(EVAL["tone"][i]))} for i in tqdm(SUB["tone"], desc=f"{name}:tone")]
    return res

def summarize(name, res):
    rows = []
    for task in ["json", "sql", "samsum"]:
        b = st.mean(REFSC[(task, "base")][i]["main"] for i in SUB[task])
        s = st.mean(REFSC[(task, "solo")][i]["main"] for i in SUB[task])
        m = st.mean(r["main"] for r in res[task])
        rows.append((task, b, s, m, (m - b) / (s - b)))
    b = st.mean(TONE_LOSS["base"][i] for i in SUB["tone"]); s = st.mean(TONE_LOSS["tone"][i] for i in SUB["tone"])
    m = st.mean(r["loss"] for r in res["tone"]); rows.append(("tone(loss)", b, s, m, (b - m) / (s - b) * -1 if False else (b - m) / (b - s)))
    print(f"\n=== {name} ===\n{'task':11} {'base':>7} {'solo':>7} {'merged':>7} {'GR':>6}")
    for t, b, s, m, g in rows: print(f"{t:11} {b:7.3f} {s:7.3f} {m:7.3f} {g:6.2f}")
    grs = [g for *_, g in rows]
    if "sql" in res: print(f"SQL EM merged: {st.mean(r['em'] for r in res['sql']):.3f}")
    print(f"AVG GR {st.mean(grs):.2f} | WORST GR {min(grs):.2f}")
    return {"rows": rows, "avg_gr": st.mean(grs), "worst_gr": min(grs)}

# ---------- M1: all 4 adapters, lambda grid ----------
os.makedirs("/kaggle/working/results/merge", exist_ok=True)
SUMMARY = {}
for lam in [1.0, 0.5, 0.25]:
    name = f"m1_all4_l{str(lam).replace('.', 'p')}"; path = f"/kaggle/working/results/merge/{name}.json"
    if os.path.exists(path):
        res = json.load(open(path))
    else:
        d = build_concat({n: lam for n in NAMES}, f"/kaggle/working/merged/{name}")
        model.load_adapter(d, adapter_name=name, is_trainable=False)
        res = eval_adapter(name)
        json.dump(res, open(path, "w"))
        model.delete_adapter(name); torch.cuda.empty_cache()
    SUMMARY[name] = summarize(name, res)


=== m1_all4_l1p0 ===
task           base    solo  merged     GR
json          0.350   0.653   0.390   0.13
sql           0.501   0.728   0.711   0.93
samsum        0.315   0.432   0.390   0.64
tone(loss)    2.832   1.663   1.437   1.19
SQL EM merged: 0.050
AVG GR 0.72 | WORST GR 0.13


m1_all4_l0p5:json:   0%|          | 0/30 [00:00<?, ?it/s]

m1_all4_l0p5:sql:   0%|          | 0/60 [00:00<?, ?it/s]

m1_all4_l0p5:samsum:   0%|          | 0/50 [00:00<?, ?it/s]

m1_all4_l0p5:tone:   0%|          | 0/150 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:2119: UserWarning: Adapter m1_all4_l0p5 was active which is now deleted. Setting active adapter to json.
  warnings.warn(


... (3 outputs left out)

m1_all4_l0p25:samsum:   0%|          | 0/50 [00:00<?, ?it/s]

m1_all4_l0p25:tone:   0%|          | 0/150 [00:00<?, ?it/s]


=== m1_all4_l0p25 ===
task           base    solo  merged     GR
json          0.350   0.653   0.641   0.96
sql           0.501   0.728   0.695   0.85
samsum        0.315   0.432   0.390   0.64
tone(loss)    2.832   1.663   1.520   1.12
SQL EM merged: 0.033
AVG GR 0.89 | WORST GR 0.64


/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:2119: UserWarning: Adapter m1_all4_l0p25 was active which is now deleted. Setting active adapter to json.
  warnings.warn(


In [14]:
res = json.load(open("/kaggle/working/results/merge/m1_all4_l0p5.json"))["sql"]
solo = REFSC[("sql", "solo")]
print("solo EM on these 60:", st.mean(solo[i]["em"] for i in SUB["sql"]))
for x in res[:6]:
    i = x["i"]
    print(f"\n#{i} | solo_em {solo[i]['em']} | merged_em {x['em']}")
    print(" REF   :", REF["sql"](EVAL["sql"][i])[:150])
    print(" MERGED:", x["pred"][:200])

solo EM on these 60: 0.13333333333333333

#0 | solo_em 0.0 | merged_em 0.0
 REF   : SELECT Character ,  Duration FROM actor
 MERGED: SELECT character ,  duration FROM actors

#1 | solo_em 0.0 | merged_em 0.0
 REF   : SELECT song_name FROM song WHERE genre_is  =  "modern" OR languages  =  "english"
 MERGED: SELECT T1.song_name FROM songs AS T1 JOIN genres AS T2 ON T1.genre_id  =  T2.genre_id WHERE T2.genre_name  =  "modern" OR T1.language  =  "English"

#2 | solo_em 1.0 | merged_em 0.0
 REF   : SELECT name FROM Aircraft ORDER BY distance LIMIT 3
 MERGED: SELECT T1.name FROM aircrafts AS T1 JOIN flights AS T2 ON T1.id  =  T2.aircraft_id ORDER BY T2.distance ASC LIMIT 3

#3 | solo_em 0.0 | merged_em 0.0
 REF   : SELECT * FROM  job_history AS T1 JOIN employees AS T2 ON T1.employee_id  =  T2.employee_id WHERE T2.salary  >=  12000
 MERGED: SELECT * FROM job_history WHERE salary >= 12000

#4 | solo_em 0.0 | merged_em 0.0
 REF   : SELECT t1.Primary_conference FROM university AS t1 JOIN basketb

In [15]:
def eval_tasks(name, tasks):
    model.set_adapter(name); res = {}
    for task in [t for t in tasks if t != "tone"]:
        out = []
        for i in tqdm(SUB[task], desc=f"{name}:{task}"):
            ex = EVAL[task][i]; p, sp, mx = PROMPT[task](ex)
            try: pred = generate(p, sp, mx)
            except torch.cuda.OutOfMemoryError: torch.cuda.empty_cache(); pred = ""
            out.append({"i": i, "pred": pred, **score_pred(task, REF[task](ex), pred)})
        res[task] = out
    if "tone" in tasks:
        res["tone"] = [{"i": i, "loss": answer_loss(*tone_ids(EVAL["tone"][i]))} for i in tqdm(SUB["tone"], desc=f"{name}:tone")]
    return res

def summarize_tasks(name, res):
    rows = []
    for task in [t for t in ["json", "sql", "samsum"] if t in res]:
        b = st.mean(REFSC[(task, "base")][i]["main"] for i in SUB[task])
        s = st.mean(REFSC[(task, "solo")][i]["main"] for i in SUB[task])
        m = st.mean(r["main"] for r in res[task]); rows.append((task, b, s, m, (m - b) / (s - b)))
    if "tone" in res:
        b = st.mean(TONE_LOSS["base"][i] for i in SUB["tone"]); s = st.mean(TONE_LOSS["tone"][i] for i in SUB["tone"])
        m = st.mean(r["loss"] for r in res["tone"]); rows.append(("tone(loss)", b, s, m, (b - m) / (b - s)))
    print(f"\n=== {name} ===\n{'task':11} {'base':>7} {'solo':>7} {'merged':>7} {'GR':>6}")
    for t, b, s, m, g in rows: print(f"{t:11} {b:7.3f} {s:7.3f} {m:7.3f} {g:6.2f}")
    grs = [g for *_, g in rows]; print(f"AVG GR {st.mean(grs):.2f} | WORST GR {min(grs):.2f}")
    return {"rows": rows, "avg_gr": st.mean(grs), "worst_gr": min(grs)}

for lam in [1.0, 0.5]:
    name = f"m1_jsontone_l{str(lam).replace('.', 'p')}"; path = f"/kaggle/working/results/merge/{name}.json"
    if os.path.exists(path):
        res = json.load(open(path))
    else:
        d = build_concat({"json": lam, "tone": lam}, f"/kaggle/working/merged/{name}")
        model.load_adapter(d, adapter_name=name, is_trainable=False)
        res = eval_tasks(name, ["json", "tone"]); json.dump(res, open(path, "w"))
        model.delete_adapter(name); torch.cuda.empty_cache()
    SUMMARY[name] = summarize_tasks(name, res)

m1_jsontone_l1p0:json:   0%|          | 0/30 [00:00<?, ?it/s]

m1_jsontone_l1p0:tone:   0%|          | 0/150 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:2119: UserWarning: Adapter m1_jsontone_l1p0 was active which is now deleted. Setting active adapter to json.
  warnings.warn(



=== m1_jsontone_l1p0 ===
task           base    solo  merged     GR
json          0.350   0.653   0.640   0.96
tone(loss)    2.832   1.663   1.626   1.03
AVG GR 0.99 | WORST GR 0.96


m1_jsontone_l0p5:json:   0%|          | 0/30 [00:00<?, ?it/s]

m1_jsontone_l0p5:tone:   0%|          | 0/150 [00:00<?, ?it/s]


=== m1_jsontone_l0p5 ===
task           base    solo  merged     GR
json          0.350   0.653   0.701   1.16
tone(loss)    2.832   1.663   1.588   1.06
AVG GR 1.11 | WORST GR 1.06


/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:2119: UserWarning: Adapter m1_jsontone_l0p5 was active which is now deleted. Setting active adapter to json.
  warnings.warn(


In [16]:
def task_vec(n, pre):
    if pre + SUF_A not in W[n]: return None
    A = W[n][pre + SUF_A].to(model.device, torch.float32)
    B = W[n][pre + SUF_B].to(model.device, torch.float32)
    return (B @ A) * SCALE                                   # asli update delta W

def trim(t, density):
    k = int(t.numel() * (1 - density))
    if k <= 0: return t
    thr = t.abs().flatten().kthvalue(k).values
    return t * (t.abs() > thr)

def build_ties(density, lam, out_dir, names=NAMES, rank=R_MERGED):
    prefixes = sorted({k[:-len(SUF_A)] for n in names for k in W[n] if k.endswith(SUF_A)})
    out, errs = {}, []
    for pre in tqdm(prefixes, desc=f"TIES d={density}"):
        tvs = [trim(t, density) for n in names if (t := task_vec(n, pre)) is not None]
        T = torch.stack(tvs)                                         # [k, out, in]
        sign = torch.sign(T.sum(0))                                  # sign election
        agree = (torch.sign(T) == sign) & (T != 0)
        merged = lam * (T * agree).sum(0) / agree.sum(0).clamp(min=1)   # disjoint mean
        U, S, V = torch.svd_lowrank(merged, q=rank + 16, niter=4)    # wapas rank-64 LoRA
        B, A = U[:, :rank] * S[:rank], V[:, :rank].T
        errs.append(((B @ A - merged).norm() / merged.norm().clamp(min=1e-8)).item())
        out[pre + SUF_A], out[pre + SUF_B] = A.contiguous().cpu(), B.contiguous().cpu()
        del T, agree, merged, tvs; torch.cuda.empty_cache()
    os.makedirs(out_dir, exist_ok=True)
    save_file(out, f"{out_dir}/adapter_model.safetensors")
    mods = sorted({p.split(".")[-1] for p in prefixes})
    LoraConfig(r=rank, lora_alpha=rank, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
               target_modules=mods, base_model_name_or_path=BASE).save_pretrained(out_dir)
    print(f"rank-{rank} reconstruction error: mean {st.mean(errs):.3f} | max {max(errs):.3f}")
    return out_dir, st.mean(errs)

ALL4 = ["json", "sql", "samsum", "tone"]
for density, lam in [(0.5, 1.0), (0.2, 1.0)]:
    name = f"m2_ties_d{str(density).replace('.', 'p')}_l{str(lam).replace('.', 'p')}"
    path = f"/kaggle/working/results/merge/{name}.json"
    if os.path.exists(path):
        res = json.load(open(path))
    else:
        d, err = build_ties(density, lam, f"/kaggle/working/merged/{name}")
        model.load_adapter(d, adapter_name=name, is_trainable=False)
        res = eval_tasks(name, ALL4); res["svd_err"] = err
        json.dump(res, open(path, "w"))
        model.delete_adapter(name); torch.cuda.empty_cache()
    SUMMARY[name] = summarize_tasks(name, res)
    if "sql" in res: print("SQL EM merged:", round(st.mean(r["em"] for r in res["sql"]), 3))

TIES d=0.5:   0%|          | 0/224 [00:00<?, ?it/s]

rank-64 reconstruction error: mean 0.367 | max 0.478


m2_ties_d0p5_l1p0:json:   0%|          | 0/30 [00:00<?, ?it/s]

m2_ties_d0p5_l1p0:sql:   0%|          | 0/60 [00:00<?, ?it/s]

m2_ties_d0p5_l1p0:samsum:   0%|          | 0/50 [00:00<?, ?it/s]

m2_ties_d0p5_l1p0:tone:   0%|          | 0/150 [00:00<?, ?it/s]

... (6 outputs left out)

m2_ties_d0p2_l1p0:samsum:   0%|          | 0/50 [00:00<?, ?it/s]

m2_ties_d0p2_l1p0:tone:   0%|          | 0/150 [00:00<?, ?it/s]


=== m2_ties_d0p2_l1p0 ===
task           base    solo  merged     GR
json          0.350   0.653   0.658   1.02
sql           0.501   0.728   0.684   0.81
samsum        0.315   0.432   0.413   0.83
tone(loss)    2.832   1.663   1.437   1.19
AVG GR 0.96 | WORST GR 0.81
SQL EM merged: 0.033


/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:2119: UserWarning: Adapter m2_ties_d0p2_l1p0 was active which is now deleted. Setting active adapter to json.
  warnings.warn(


In [17]:
GEN = torch.Generator(device=model.device).manual_seed(42)

def magprune(t, density, eps=0.1):
    """DELLA MagPrune: bade |value| ka keep-probability zyada; survivors ko 1/p se rescale."""
    r = t.abs().argsort(dim=1).argsort(dim=1).float() / max(1, t.shape[1] - 1)   # row ke andar rank, 0..1
    p_keep = (density + eps * (r - 0.5)).clamp(1e-3, 1.0)
    keep = torch.rand(t.shape, generator=GEN, device=t.device) < p_keep
    out = t * keep / p_keep
    del r, p_keep, keep
    return out

def build_della(density, lam, out_dir, names=NAMES, rank=R_MERGED, eps=0.1):
    prefixes = sorted({k[:-len(SUF_A)] for n in names for k in W[n] if k.endswith(SUF_A)})
    out, errs = {}, []
    for pre in tqdm(prefixes, desc=f"DELLA d={density}"):
        tvs = [magprune(t, density, eps) for n in names if (t := task_vec(n, pre)) is not None]
        T = torch.stack(tvs)
        sign = torch.sign(T.sum(0))
        agree = (torch.sign(T) == sign) & (T != 0)
        merged = lam * (T * agree).sum(0) / agree.sum(0).clamp(min=1)
        U, S, V = torch.svd_lowrank(merged, q=rank + 16, niter=4)
        B, A = U[:, :rank] * S[:rank], V[:, :rank].T
        errs.append(((B @ A - merged).norm() / merged.norm().clamp(min=1e-8)).item())
        out[pre + SUF_A], out[pre + SUF_B] = A.contiguous().cpu(), B.contiguous().cpu()
        del T, agree, merged, tvs; torch.cuda.empty_cache()
    os.makedirs(out_dir, exist_ok=True)
    save_file(out, f"{out_dir}/adapter_model.safetensors")
    mods = sorted({p.split(".")[-1] for p in prefixes})
    LoraConfig(r=rank, lora_alpha=rank, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
               target_modules=mods, base_model_name_or_path=BASE).save_pretrained(out_dir)
    print(f"rank-{rank} reconstruction error: mean {st.mean(errs):.3f} | max {max(errs):.3f}")
    return out_dir, st.mean(errs)

for density, lam in [(0.2, 1.0), (0.5, 1.0)]:
    name = f"m3_della_d{str(density).replace('.', 'p')}_l{str(lam).replace('.', 'p')}"
    path = f"/kaggle/working/results/merge/{name}.json"
    if os.path.exists(path):
        res = json.load(open(path))
    else:
        d, err = build_della(density, lam, f"/kaggle/working/merged/{name}")
        model.load_adapter(d, adapter_name=name, is_trainable=False)
        res = eval_tasks(name, ALL4); res["svd_err"] = err
        json.dump(res, open(path, "w"))
        model.delete_adapter(name); torch.cuda.empty_cache()
    SUMMARY[name] = summarize_tasks(name, res)
    if "sql" in res: print("SQL EM merged:", round(st.mean(r["em"] for r in res["sql"]), 3))

DELLA d=0.2:   0%|          | 0/224 [00:00<?, ?it/s]

rank-64 reconstruction error: mean 0.849 | max 0.869


m3_della_d0p2_l1p0:json:   0%|          | 0/30 [00:00<?, ?it/s]

m3_della_d0p2_l1p0:sql:   0%|          | 0/60 [00:00<?, ?it/s]

m3_della_d0p2_l1p0:samsum:   0%|          | 0/50 [00:00<?, ?it/s]

m3_della_d0p2_l1p0:tone:   0%|          | 0/150 [00:00<?, ?it/s]

... (6 outputs left out)

m3_della_d0p5_l1p0:samsum:   0%|          | 0/50 [00:00<?, ?it/s]

m3_della_d0p5_l1p0:tone:   0%|          | 0/150 [00:00<?, ?it/s]


=== m3_della_d0p5_l1p0 ===
task           base    solo  merged     GR
json          0.350   0.653   0.601   0.83
sql           0.501   0.728   0.713   0.93
samsum        0.315   0.432   0.396   0.69
tone(loss)    2.832   1.663   1.505   1.14
AVG GR 0.90 | WORST GR 0.69
SQL EM merged: 0.033


/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:2119: UserWarning: Adapter m3_della_d0p5_l1p0 was active which is now deleted. Setting active adapter to json.
  warnings.warn(


In [18]:
def core_factors(pre, names=NAMES):
    """Returns U_B (out x R), V_A (in x R), aur har adapter ka core matrix (R x R)."""
    As, Bs = [], []
    for n in names:
        if pre + SUF_A in W[n]:
            As.append(W[n][pre + SUF_A].to(model.device, torch.float32))
            Bs.append(W[n][pre + SUF_B].to(model.device, torch.float32) * SCALE)
    U_B, _ = torch.linalg.qr(torch.cat(Bs, 1))        # out x R
    V_A, _ = torch.linalg.qr(torch.cat(As, 0).T)      # in  x R
    cores = [U_B.T @ B @ A @ V_A for A, B in zip(As, Bs)]
    # lossless check: core se wapas asli update
    A0, B0 = As[0], Bs[0]
    err = ((U_B @ cores[0] @ V_A.T) - B0 @ A0).norm() / (B0 @ A0).norm()
    return U_B, V_A, cores, err.item()

def tsv_merge(cores, alpha):
    Us, Ss, Vs = [], [], []
    k = max(1, cores[0].shape[0] // len(cores))
    for C in cores:
        U, S, Vh = torch.linalg.svd(C, full_matrices=False)
        Us.append(U[:, :k]); Ss.append(S[:k]); Vs.append(Vh[:k].T)
    U, S, V = torch.cat(Us, 1), torch.cat(Ss), torch.cat(Vs, 1)
    P, _, Qh = torch.linalg.svd(U, full_matrices=False); U_o = P @ Qh      # decorrelate
    P, _, Qh = torch.linalg.svd(V, full_matrices=False); V_o = P @ Qh
    return alpha * U_o @ torch.diag(S) @ V_o.T

def iso_merge(cores, alpha):
    U, S, Vh = torch.linalg.svd(sum(cores), full_matrices=False)
    return alpha * U @ torch.diag(torch.full_like(S, S.mean())) @ Vh        # flat spectrum

def build_core(method, alpha, out_dir, names=NAMES, rank=R_MERGED):
    prefixes = sorted({k[:-len(SUF_A)] for n in names for k in W[n] if k.endswith(SUF_A)})
    out, errs = {}, []
    for pre in tqdm(prefixes, desc=f"core-{method}"):
        U_B, V_A, cores, err = core_factors(pre, names); errs.append(err)
        M = tsv_merge(cores, alpha) if method == "tsv" else iso_merge(cores, alpha)
        B, A = U_B @ M, V_A.T                          # out x R, R x in  (exact)
        pad = rank - A.shape[0]
        if pad:
            A = torch.cat([A, A.new_zeros(pad, A.shape[1])], 0)
            B = torch.cat([B, B.new_zeros(B.shape[0], pad)], 1)
        out[pre + SUF_A], out[pre + SUF_B] = A.contiguous().cpu(), B.contiguous().cpu()
    os.makedirs(out_dir, exist_ok=True)
    save_file(out, f"{out_dir}/adapter_model.safetensors")
    mods = sorted({p.split(".")[-1] for p in prefixes})
    LoraConfig(r=rank, lora_alpha=rank, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
               target_modules=mods, base_model_name_or_path=BASE).save_pretrained(out_dir)
    print(f"core projection error (should be ~0): max {max(errs):.2e}")
    return out_dir

for method, alpha in [("tsv", 1.0), ("tsv", 0.5), ("iso", 1.0), ("iso", 0.5)]:
    tag = "m5" if method == "tsv" else "m6"
    name = f"{tag}_core_{method}_a{str(alpha).replace('.', 'p')}"
    path = f"/kaggle/working/results/merge/{name}.json"
    if os.path.exists(path):
        res = json.load(open(path))
    else:
        d = build_core(method, alpha, f"/kaggle/working/merged/{name}")
        model.load_adapter(d, adapter_name=name, is_trainable=False)
        res = eval_tasks(name, ALL4); json.dump(res, open(path, "w"))
        model.delete_adapter(name); torch.cuda.empty_cache()
    SUMMARY[name] = summarize_tasks(name, res)
    if "sql" in res: print("SQL EM merged:", round(st.mean(r["em"] for r in res["sql"]), 3))

core-tsv:   0%|          | 0/224 [00:00<?, ?it/s]

core projection error (should be ~0): max 4.30e-07


m5_core_tsv_a1p0:json:   0%|          | 0/30 [00:00<?, ?it/s]

m5_core_tsv_a1p0:sql:   0%|          | 0/60 [00:00<?, ?it/s]

m5_core_tsv_a1p0:samsum:   0%|          | 0/50 [00:00<?, ?it/s]

m5_core_tsv_a1p0:tone:   0%|          | 0/150 [00:00<?, ?it/s]

... (22 outputs left out)

m6_core_iso_a0p5:samsum:   0%|          | 0/50 [00:00<?, ?it/s]

m6_core_iso_a0p5:tone:   0%|          | 0/150 [00:00<?, ?it/s]


=== m6_core_iso_a0p5 ===
task           base    solo  merged     GR
json          0.350   0.653   0.002  -1.15
sql           0.501   0.728   0.674   0.76
samsum        0.315   0.432   0.362   0.41
tone(loss)    2.832   1.663   1.635   1.02
AVG GR 0.26 | WORST GR -1.15
SQL EM merged: 0.017


/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:2119: UserWarning: Adapter m6_core_iso_a0p5 was active which is now deleted. Setting active adapter to json.
  warnings.warn(


In [ ]:
@torch.no_grad()
def adapter_norm(n):
    """||Delta W||_F poore adapter ka, bina bada matrix banaye: ||BA||^2 = sum((B^T B) * (A A^T))."""
    tot = 0.0
    for k in W[n]:
        if not k.endswith(SUF_A): continue
        pre = k[:-len(SUF_A)]
        A = W[n][pre + SUF_A].to(model.device, torch.float32)
        B = W[n][pre + SUF_B].to(model.device, torch.float32) * SCALE
        tot += ((B.T @ B) * (A @ A.T)).sum().item()
    return tot ** 0.5

NORMS = {n: adapter_norm(n) for n in NAMES}
G = float(torch.tensor(list(NORMS.values())).log().mean().exp())        # geometric mean
NB_W = {n: G / NORMS[n] for n in NAMES}                                   # sabko same size par
print("norms:", {n: round(v, 2) for n, v in NORMS.items()})
print("balance weights:", {n: round(v, 2) for n, v in NB_W.items()})

def core_factors_w(pre, weights):
    As, Bs = [], []
    for n, w in weights.items():
        if pre + SUF_A in W[n]:
            As.append(W[n][pre + SUF_A].to(model.device, torch.float32))
            Bs.append(W[n][pre + SUF_B].to(model.device, torch.float32) * SCALE * w)
    U_B, _ = torch.linalg.qr(torch.cat(Bs, 1)); V_A, _ = torch.linalg.qr(torch.cat(As, 0).T)
    return U_B, V_A, [U_B.T @ B @ A @ V_A for A, B in zip(As, Bs)]

def build_core_tsv_w(weights, alpha, out_dir, rank=R_MERGED):
    prefixes = sorted({k[:-len(SUF_A)] for n in weights for k in W[n] if k.endswith(SUF_A)})
    out = {}
    for pre in tqdm(prefixes, desc="core-tsv-nb"):
        U_B, V_A, cores = core_factors_w(pre, weights)
        B, A = U_B @ tsv_merge(cores, alpha), V_A.T
        pad = rank - A.shape[0]
        if pad:
            A = torch.cat([A, A.new_zeros(pad, A.shape[1])], 0)
            B = torch.cat([B, B.new_zeros(B.shape[0], pad)], 1)
        out[pre + SUF_A], out[pre + SUF_B] = A.contiguous().cpu(), B.contiguous().cpu()
    os.makedirs(out_dir, exist_ok=True)
    save_file(out, f"{out_dir}/adapter_model.safetensors")
    mods = sorted({p.split(".")[-1] for p in prefixes})
    LoraConfig(r=rank, lora_alpha=rank, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
               target_modules=mods, base_model_name_or_path=BASE).save_pretrained(out_dir)
    return out_dir

CONFIGS = {
    "m7_nb_concat_l0p5":  lambda d: build_concat({n: 0.5 * NB_W[n] for n in NAMES}, d),
    "m7_nb_coretsv_a0p5": lambda d: build_core_tsv_w(NB_W, 0.5, d),
}
for name, builder in CONFIGS.items():
    path = f"/kaggle/working/results/merge/{name}.json"
    if os.path.exists(path):
        res = json.load(open(path))
    else:
        d = builder(f"/kaggle/working/merged/{name}")
        model.load_adapter(d, adapter_name=name, is_trainable=False)
        res = eval_tasks(name, ALL4); json.dump(res, open(path, "w"))
        model.delete_adapter(name); torch.cuda.empty_cache()
    SUMMARY[name] = summarize_tasks(name, res)
    if "sql" in res: print("SQL EM merged:", round(st.mean(r["em"] for r in res["sql"]), 3))

In [ ]:
import random
from datasets import load_dataset

DEVN = {"json": 20, "sql": 30, "samsum": 30, "tone": 30}
DEV = {}

# JSON: Phase 1 ne shuffle(42) ke pehle 71 liye the -> agle 20 dev
pj = load_dataset("paraloq/json_data_extraction")["train"].shuffle(seed=42)
DEV["json"] = pj.select(range(71, 71 + DEVN["json"]))
assert not set(DEV["json"]["text"]) & set(EVAL["json"]["text"]), "JSON overlap!"

# SQL: task107 ka valid split (test se alag)
DEV["sql"] = load_dataset("Lots-of-LoRAs/task107_splash_question_to_sql")["valid"].select(range(DEVN["sql"]))

# SAMSum: validation split, adapter ke training dialogues hata ke
sv = load_dataset("knkarthick/samsum")["validation"]
sv = sv.filter(lambda e: norm(ni_dialogue(e["dialogue"]))[:150] not in ni_text)
DEV["samsum"] = sv.select(range(DEVN["samsum"]))

# Tone: Phase 1 jaisa hi construction, par 150 ke BAAD wali conversations
arp = load_dataset("arpieb/consumer-electronics-support")
arp = arp[list(arp.keys())[0]]
cands = []
for row in arp:
    t = row["turns"]
    if len(t) < 2 or t[-1]["role"] != "agent" or not any(x["role"] == "customer" for x in t[:-1]): continue
    cands.append({"id": row["record_id"], "subject": row["scenario"],
                  "body": "\n".join(f"{x['role'].capitalize()}: {x['content']}" for x in t[:-1]),
                  "answer": t[-1]["content"]})
random.Random(42).shuffle(cands)
assert [c["id"] for c in cands[:150]] == list(EVAL["tone"]["id"]), "tone order mismatch - Phase 1 se alag"
DEV["tone"] = cands[150:150 + DEVN["tone"]]

# (prompt_ids, answer_ids) pairs - loss ke liye
def pair(task, ex):
    if task == "tone": return tone_ids(ex)
    if task == "samsum": p = tok(samsum_prompt(ex), add_special_tokens=False)["input_ids"]; a = ex["summary"]
    elif task == "sql":  p = tok(chat(ex["input"]))["input_ids"]; a = REF["sql"](ex)
    else:                p = tok(PROMPT["json"](ex)[0])["input_ids"]; a = ex["item"]
    return p, tok(a.strip(), add_special_tokens=False)["input_ids"] + [tok.eos_token_id]

DEV_IDS = {t: [x for x in (pair(t, ex) for ex in DEV[t]) if len(x[0]) + len(x[1]) <= 3000] for t in DEV}
print({t: len(v) for t, v in DEV_IDS.items()})

In [ ]:
from peft.tuners.lora import LoraLayer

def set_mix(weights):
    """Diye gaye adapters ek saath ON, har ek ka scale = weight (exact weighted sum)."""
    model.base_model.set_adapter(list(weights))
    for m in model.modules():
        if isinstance(m, LoraLayer):
            for n in NAMES:
                if n in m.lora_A: m.set_scale(n, weights.get(n, 1.0))

def reset_mix():
    set_mix({n: 1.0 for n in NAMES}); model.set_adapter("json")

@torch.no_grad()
def dev_loss(weights=None):
    """weights=None -> base model. Returns {task: mean answer loss}."""
    out = {}
    for t, pairs_ in DEV_IDS.items():
        ls = []
        for p, a in pairs_:
            if weights is None:
                with model.disable_adapter(): ls.append(answer_loss(p, a))
            else:
                ls.append(answer_loss(p, a))
        out[t] = st.mean(ls)
    return out

# --- sanity: mix(0.5 sab) == m1_all4_l0p5 merged file ---
probe_ids = tok(probe, return_tensors="pt").to(model.device)
set_mix({n: 0.5 for n in NAMES})
with torch.no_grad(): a = model(**probe_ids).logits[0, -1].float()
model.load_adapter("/kaggle/working/merged/m1_all4_l0p5", adapter_name="chk", is_trainable=False)
model.set_adapter("chk")
with torch.no_grad(): b = model(**probe_ids).logits[0, -1].float()
model.delete_adapter("chk")
print(f"SANITY mix vs merged file: max|Δlogit| {(a - b).abs().max().item():.4f} | same top: {a.argmax() == b.argmax()}")

# --- dev reference losses ---
DEV_BASE = dev_loss(None)
DEV_SOLO = {}
for t in NAMES:
    set_mix({t: 1.0}); DEV_SOLO[t] = dev_loss({t: 1.0})[t]
reset_mix()
print("DEV base:", {k: round(v, 3) for k, v in DEV_BASE.items()})
print("DEV solo:", {k: round(v, 3) for k, v in DEV_SOLO.items()})

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nevergrad"], check=True)
import nevergrad as ng, numpy as np

LOG = "/kaggle/working/results/merge/m4_search_log.json"
trials = []

def objective(w):
    weights = {n: float(x) for n, x in zip(NAMES, w)}
    set_mix(weights)
    L = dev_loss(weights)
    gap = {t: (L[t] - DEV_SOLO[t]) / (DEV_BASE[t] - DEV_SOLO[t]) for t in NAMES}
    score = st.mean(gap.values()) + 0.5 * max(gap.values())
    trials.append({"w": weights, "gap": gap, "score": score})
    json.dump(trials, open(LOG, "w"), indent=1)          # har try save
    return score

param = ng.p.Array(init=np.full(len(NAMES), 0.5)).set_bounds(0.0, 1.5)   # start = M1 (0.5 sab)
opt = ng.optimizers.NGOpt(parametrization=param, budget=30)
for _ in tqdm(range(opt.budget), desc="M4 search"):
    x = opt.ask(); opt.tell(x, objective(x.value))
reset_mix()

trials_sorted = sorted(trials, key=lambda r: r["score"])
print("start (sab 0.5):", round(trials[0]["score"], 3), {k: round(v, 2) for k, v in trials[0]["gap"].items()})
print("\nTOP 5:")
for r in trials_sorted[:5]:
    print(round(r["score"], 3), "| w:", {k: round(v, 2) for k, v in r["w"].items()},
          "| gap:", {k: round(v, 2) for k, v in r["gap"].items()})
M4_W = trials_sorted[0]["w"]
print("\nBEST WEIGHTS:", {k: round(v, 3) for k, v in M4_W.items()})

In [ ]:
name = "m4_lorahub"
path = f"/kaggle/working/results/merge/{name}.json"
if os.path.exists(path):
    res = json.load(open(path))
else:
    d = build_concat(M4_W, f"/kaggle/working/merged/{name}")
    model.load_adapter(d, adapter_name=name, is_trainable=False)
    res = eval_tasks(name, ALL4); res["weights"] = M4_W
    json.dump(res, open(path, "w"))
    model.delete_adapter(name); torch.cuda.empty_cache()
SUMMARY[name] = summarize_tasks(name, res)
print("SQL EM merged:", round(st.mean(r["em"] for r in res["sql"]), 3))

In [19]:
# 1) check: Cell 22 ke liye zaroori cheezein ready hain?
need = ["build_concat", "build_core", "eval_tasks", "summarize_tasks", "PROMPT", "REF",
        "REFSC", "samsum_prompt", "EVAL", "SUB", "TONE_LOSS", "tok", "model", "NAMES"]
missing = [n for n in need if n not in globals()]
assert not missing, f"MISSING: {missing} - ye bhejo"
print("sab ready")

# 2) M4 ke learned weights (kal ke search se)
M4_W = {"json": 0.538, "tone": 0.377, "sql": 0.941, "samsum": 0.674}

# 3) held-out ke liye merged folders (jo nahi hain, wo ban jayenge)
if not os.path.exists("/kaggle/working/merged/m4_lorahub"):
    build_concat(M4_W, "/kaggle/working/merged/m4_lorahub")
if not os.path.exists("/kaggle/working/merged/m1_all4_l0p5"):
    build_concat({n: 0.5 for n in NAMES}, "/kaggle/working/merged/m1_all4_l0p5")
if not os.path.exists("/kaggle/working/merged/m5_core_tsv_a0p5"):
    build_core("tsv", 0.5, "/kaggle/working/merged/m5_core_tsv_a0p5")
print("merged folders:", sorted(os.listdir("/kaggle/working/merged")))

sab ready
merged folders: ['_single_json', '_single_samsum', '_single_sql', '_single_tone', 'm1_all4_l0p25', 'm1_all4_l0p5', 'm1_all4_l1p0', 'm1_jsontone_l0p5', 'm1_jsontone_l1p0', 'm2_ties_d0p2_l1p0', 'm2_ties_d0p5_l1p0', 'm3_della_d0p2_l1p0', 'm3_della_d0p5_l1p0', 'm4_lorahub', 'm5_core_tsv_a0p5', 'm5_core_tsv_a1p0', 'm6_core_iso_a0p5', 'm6_core_iso_a1p0']


In [20]:
SUB_DEVSET = {k: list(v) for k, v in SUB.items()}            # jo ab tak use hua
HOLD = {"json":   [i for i in range(len(EVAL["json"])) if i != 56 and i not in SUB_DEVSET["json"]],
        "sql":    [i for i in range(len(EVAL["sql"]))    if i not in SUB_DEVSET["sql"]],
        "samsum": [i for i in range(len(EVAL["samsum"])) if i not in SUB_DEVSET["samsum"]]}
print({k: len(v) for k, v in HOLD.items()})

FINAL = {"m4_lorahub": "/kaggle/working/merged/m4_lorahub",
         "m1_all4_l0p5": "/kaggle/working/merged/m1_all4_l0p5",
         "m5_core_tsv_a0p5": "/kaggle/working/merged/m5_core_tsv_a0p5"}
FINAL_SUM = {}
SUB = {**SUB_DEVSET, **HOLD}                                  # eval/summarize ab held-out rows par
try:
    for name, d in FINAL.items():
        path = f"/kaggle/working/results/merge/{name}_HOLDOUT.json"
        if os.path.exists(path):
            res = json.load(open(path))
        else:
            assert os.path.exists(d), f"merged folder missing: {d}"
            model.load_adapter(d, adapter_name=name, is_trainable=False)
            res = eval_tasks(name, ["json", "sql", "samsum"]); json.dump(res, open(path, "w"))
            model.delete_adapter(name); torch.cuda.empty_cache()
        FINAL_SUM[name] = summarize_tasks(name + " [HELD-OUT]", res)
        print("SQL EM merged:", round(st.mean(r["em"] for r in res["sql"]), 3))
finally:
    SUB = SUB_DEVSET                                          # wapas restore

print("\n=== FINAL (held-out) ===")
for name, s in FINAL_SUM.items():
    print(f"{name:20} AVG GR {s['avg_gr']:.2f} | WORST GR {s['worst_gr']:.2f}")

{'json': 40, 'sql': 144, 'samsum': 100}


m4_lorahub:json:   0%|          | 0/40 [00:00<?, ?it/s]

m4_lorahub:sql:   0%|          | 0/144 [00:00<?, ?it/s]

m4_lorahub:samsum:   0%|          | 0/100 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:2119: UserWarning: Adapter m4_lorahub was active which is now deleted. Setting active adapter to json.
  warnings.warn(



=== m4_lorahub [HELD-OUT] ===
task           base    solo  merged     GR
json          0.383   0.668   0.610   0.80
sql           0.499   0.741   0.714   0.89
samsum        0.310   0.411   0.401   0.91
AVG GR 0.86 | WORST GR 0.80
SQL EM merged: 0.062


... (4 outputs left out)

m5_core_tsv_a0p5:json:   0%|          | 0/40 [00:00<?, ?it/s]

m5_core_tsv_a0p5:sql:   0%|          | 0/144 [00:00<?, ?it/s]

m5_core_tsv_a0p5:samsum:   0%|          | 0/100 [00:00<?, ?it/s]


=== m5_core_tsv_a0p5 [HELD-OUT] ===
task           base    solo  merged     GR
json          0.383   0.668   0.526   0.50
sql           0.499   0.741   0.700   0.83
samsum        0.310   0.411   0.386   0.76
AVG GR 0.70 | WORST GR 0.50
SQL EM merged: 0.021

=== FINAL (held-out) ===
m4_lorahub           AVG GR 0.86 | WORST GR 0.80
m1_all4_l0p5         AVG GR 0.84 | WORST GR 0.80
m5_core_tsv_a0p5     AVG GR 0.70 | WORST GR 0.50


In [21]:
import shutil, os, json
os.makedirs("/kaggle/working/phase2", exist_ok=True)
shutil.copytree("/kaggle/working/results", "/kaggle/working/phase2/results", dirs_exist_ok=True)
shutil.copytree("/kaggle/working/scores",  "/kaggle/working/phase2/scores",  dirs_exist_ok=True)
json.dump({"M4_W": M4_W}, open("/kaggle/working/phase2/weights.json", "w"), indent=2)
json.dump({k: {"avg_gr": v["avg_gr"], "worst_gr": v["worst_gr"], "rows": v["rows"]} for k, v in FINAL_SUM.items()},
          open("/kaggle/working/phase2/final_holdout_summary.json", "w"), indent=2, default=str)
shutil.make_archive("/kaggle/working/phase2_results", "zip", "/kaggle/working/phase2")
print(round(os.path.getsize("/kaggle/working/phase2_results.zip") / 1e6, 1), "MB")

0.3 MB
